### Note: Not all Ollama models support tool calling

`create_agent` (and any tool-using agent) requires the underlying model to have **native tool-calling support** — the model must understand a `tools` field in the request and reply with structured `tool_calls`, not just plain text.

**Models that do NOT support this:** Gemma 2, Gemma 3 (no dedicated tool-call tokens — trying to use them here throws `does not support tools` from Ollama).

**Models confirmed to support tools:** Llama 3.1+, Qwen 2.5+, Qwen3, Mistral, Command-R.

Recommended for this project:
```bash
ollama pull qwen3:8b
# or
ollama pull llama3.1:8b
```

**Before using any model in an agent, verify it supports tools:**
```bash
ollama show <model_name>
```
Check that `tools` appears under the **Capabilities** section. If it's missing, swap the model rather than debugging the agent code — it's not a code issue.

In [5]:
# %pip install -U langchain-community mypy_extensions

In [1]:
import langchain_core
from langchain_classic.agents import AgentType, initialize_agent, load_tools
from langchain_classic.chains.sequential import SequentialChain
from langchain_classic.chains.llm import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama
from config_loader import load_config_as_dict

In [2]:
llm_model: ChatOllama = None

def llm_initializer():
    global llm_model
    if llm_model:
        print("(*) LLM Model Already Initialized.")
        return
    llm_config = load_config_as_dict()['llm']
    llm_config['model'] = 'qwen3:8b'  # Didn't change the config.yaml file so changed here
    llm_model = ChatOllama(
        model=llm_config['model'],
        temperature=llm_config['temperature'],
        num_predict=llm_config['max_tokens'],  # Ollama's equivalent of Groq's max_tokens
        num_ctx=llm_config['num_ctx'],
        reasoning=llm_config['reasoning'],
        keep_alive=llm_config['keep_alive'],
    )

In [3]:
# %pip install wikipedia numexpr

In [4]:
from langchain_core.tools import tool
from langchain_community.agent_toolkits.load_tools import load_tools
import numexpr
from datetime import datetime
import pytz


@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression, e.g. "2 + 2" or "3**2". Use standard Python math syntax (** for power, not ^)."""
    try:
        result = numexpr.evaluate(expression).item()
        return str(result)
    except Exception as e:
        return f"Error evaluating expression: {e}"


@tool
def current_datetime(region: str = 'Asia/Kolkata') -> str:
    """Get the current date and time for a given region.
    Returns:
        The current date and time in that region as MM/DD/YYYY HH:MM AM/PM.
    """
    try:
        tz = pytz.timezone(region)
        return datetime.now(tz).strftime("%m/%d/%Y %I:%M %p")
    except pytz.exceptions.UnknownTimeZoneError:
        return f"Error: '{region}' is not a recognized timezone. Use an IANA name like 'Asia/Kolkata' or 'America/New_York'."
    except Exception as e:
        return f"Error retrieving datetime for region {region!r}: {e}"

In [5]:
tools_name = ['ddg-search'] #, 'llm-math']  # wikipedia
llm_initializer()
llm_tools = load_tools(tools_name, llm=llm_model)
llm_tools += [calculator, current_datetime]

In [6]:
llm_agent = initialize_agent(
    llm_tools,
    llm_model,
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)
# query = "When was Elon Musk born? What is his age right now in 2026?"
# query = "Evaluate this expression: x^3, where x belongs to a set of {1, 2, 3}"
query = "When was Elon Musk born? What is his age right now?"
# llm_agent.invoke(query)  # was throwing errors related to some function call
llm_agent.invoke({"input": [{"role": "user", "content": query}]})

C:\Users\ShubhanshuJha\AppData\Local\Temp\ipykernel_43116\3928863115.py:1: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  llm_agent = initialize_agent(




> Entering new AgentExecutor chain...
Thought: I need to find Elon Musk's birth date and calculate his current age. First, I will search for his birth date.
Action:
```
{
  "action": "duckduckgo_search",
  "action_input": "Elon Musk birth date"
}
```

Observation: Elon Musk, Tesla and SpaceX CEO, is a visionary entrepreneur shaping technology, space travel, and clean energy with bold innovation. Elon Musk is a South African -born American entrepreneur who cofounded the electronic payment firm PayPal and formed SpaceX, maker of launch vehicles and spacecraft. Elon Musk is #1 on Forbes' 2026 America's Richest Immigrants list. Read more about Elon Musk, their experience, their asset summary, and more here. Learn interesting facts about Elon Musk (Entrepreneur). Discover Elon Musk age, birthday, birthplace, horoscope, wiki, biography, before fame, family and social media. Who is Elon Musk? Elon Musk is an engineer, entrepreneur, and investor. He is a co-founder of PayPal, co-founder and 

{'input': [{'role': 'user',
   'content': 'When was Elon Musk born? What is his age right now?'}],
 'output': 'Elon Musk was born on June 28, 1971, and he is 55 years old as of August 30, 2026.'}

In [5]:
2026 - 1971

55

In [7]:
def build_system_prompt(tools: list) -> str:
    """
    Build the agent's system prompt dynamically from whatever tools are
    currently registered, so adding or removing tools never requires
    hand-editing this prompt again.
    """
    tool_lines = "\n".join(
        f"- {t.name}: {t.description.strip().splitlines()[0]}" for t in tools
    )
    return f"""You are a helpful assistant with access to the following tools:
    {tool_lines}

    Guidelines:
    - Choose whichever tool best fits the question, based on its description above — do not limit yourself to a fixed mapping of tool-to-task-type.
    - Use a tool whenever it would make your answer more accurate or current, especially for facts that could be outdated, time-sensitive, numeric, or easy to get wrong from memory alone.
    - Before giving your final answer, use the tools available to verify or cross-check any facts, dates, or calculations you are unsure of — do not rely solely on your own knowledge if a tool can confirm it.
    - If none of the tools are relevant to the question, answer directly without forcing tool use.
    - Answer concisely once you are confident in the accuracy of your response.
    """

In [8]:
from langchain.agents import create_agent


# SYSTEM_PROMPT = """
# You are a helpful assistant with access to tools: web search, Wikipedia,
# and a calculator. Use Wikipedia for stable facts (biographies, history, definitions),
# web search for anything current or time-sensitive, and the calculator for any
# arithmetic — don't compute math yourself. Answer concisely and only use a tool when you actually need it.
# """
SYSTEM_PROMPT = build_system_prompt(llm_tools)

llm_agent = create_agent(
    tools=llm_tools,
    model=llm_model,
    system_prompt=SYSTEM_PROMPT
)

query = "When was Elon Musk born? What is his age right now in 2026?"
llm_response = llm_agent.invoke({"messages": [{"role": "user", "content": query}]})
llm_response

{'messages': [HumanMessage(content='When was Elon Musk born? What is his age right now in 2026?', additional_kwargs={}, response_metadata={}, id='e74a6bd0-3678-4914-8307-b1cc8b2d624d'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-08-30T15:48:41.8555336Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2499799300, 'load_duration': 20654600, 'prompt_eval_count': 563, 'prompt_eval_duration': 603155000, 'eval_count': 26, 'eval_duration': 892506000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--01a0535b-c936-7e03-8028-c0ec2a38ad82-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'Elon Musk birth date'}, 'id': '7f9964e3-9f17-4992-b190-4f070b7c9b1a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 563, 'output_tokens': 26, 'total_tokens': 589}),
  ToolMessage(content="25 Nov 2025 · एलन रीव मस्क 28 जून 1971 (1971-06-28) (आयु 55) प्रिटोरिया, त

In [9]:
type(llm_response)

dict

In [10]:
llm_response['messages']

[HumanMessage(content='When was Elon Musk born? What is his age right now in 2026?', additional_kwargs={}, response_metadata={}, id='e74a6bd0-3678-4914-8307-b1cc8b2d624d'),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-08-30T15:48:41.8555336Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2499799300, 'load_duration': 20654600, 'prompt_eval_count': 563, 'prompt_eval_duration': 603155000, 'eval_count': 26, 'eval_duration': 892506000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--01a0535b-c936-7e03-8028-c0ec2a38ad82-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'Elon Musk birth date'}, 'id': '7f9964e3-9f17-4992-b190-4f070b7c9b1a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 563, 'output_tokens': 26, 'total_tokens': 589}),
 ToolMessage(content="25 Nov 2025 · एलन रीव मस्क 28 जून 1971 (1971-06-28) (आयु 55) प्रिटोरिया, त्रांसवाल, दक्षि

In [11]:
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama


class ResponseCheck(BaseModel):
    is_correct: bool = Field(description="True if the response fully and correctly answers the query.")
    corrected_response: str = Field(
        description="Corrected, complete answer if is_correct is False; otherwise the original response unchanged."
    )
    reasoning: str = Field(description="Brief explanation of what was missing or wrong, if anything.")


def rectify_response(query: str, response: str, llm: ChatOllama) -> str:
    """
    Check whether `response` fully and correctly answers `query`, and
    return a corrected version if it doesn't.
    """
    verification_prompt = f"""You are a strict reviewer. Check whether the RESPONSE fully and correctly answers every part of the USER QUERY.

    USER QUERY:
    {query}

    RESPONSE:
    {response}

    If the response is complete and correct, set is_correct=True and repeat the response unchanged in corrected_response.
    If it is incomplete, incorrect, or misses part of a multi-part question, set is_correct=False and provide a corrected, complete answer in corrected_response.
    """
    structured_llm = llm.with_structured_output(ResponseCheck)
    result: ResponseCheck = structured_llm.invoke(verification_prompt)

    if not result.is_correct:
        print(f"(*) Response rectified. Reason: {result.reasoning}")

    return result.corrected_response

In [12]:
raw_response = llm_response["messages"][-1].content
final_response = rectify_response(query=query, response=raw_response, llm=llm_model)

In [13]:
final_response

'Elon Musk was born on June 28, 1971. As of 2026, his age is 55 years old.'

In [19]:
def ask_llm(user_query: str):
    global llm_model
    raw_llm_response = llm_agent.invoke({"messages": [{"role": "user", "content": user_query}]})
    print("(*) Received raw response from the LLM. Rectifying the response...")
    final_llm_response = rectify_response(query=user_query, response=raw_llm_response["messages"][-1].content, llm=llm_model)
    print("(*) Response rectified. Returning the final LLM response.\n")
    return final_llm_response.strip()

### Memory

In [17]:
query = "What is the role of an AI Data Engineer?"
print(ask_llm(user_query=query))

(*) Received raw response from the LLM. Rectifying the response...
(*) Response rectified. Returning the final LLM response.
An AI Data Engineer is responsible for designing, building, and maintaining the infrastructure and systems that enable the collection, storage, processing, and analysis of data used to train and deploy AI models. Their role includes:

1. **Data Pipeline Development**: Creating and maintaining data pipelines to efficiently move and process large volumes of data from various sources to machine learning models.
2. **Data Storage and Management**: Designing and managing databases and data warehouses to store structured and unstructured data.
3. **Data Processing and Transformation**: Ensuring data is cleaned, transformed, and formatted appropriately for use in AI models.
4. **Integration with AI/ML Tools**: Working with tools and frameworks like TensorFlow, PyTorch, and cloud platforms (e.g., AWS, Azure, GCP) to support AI model development and deployment.
5. **Colla

In [18]:
query = "Who is most powerful person on the Earth right now?"
print(ask_llm(user_query=query))

(*) Received raw response from the LLM. Rectifying the response...
(*) Response rectified. Reason: The original response was incomplete because it did not provide a definitive answer to the question 'Who is the most powerful person on the Earth right now?' It listed several individuals but did not identify a single person as the most powerful. The corrected response addresses this by acknowledging the subjectivity of the term and then providing a specific example of the most powerful political figure (Joe Biden) and the most powerful individual in terms of economic and technological influence (Elon Musk), while also offering further assistance.
(*) Response rectified. Returning the final LLM response.
The concept of 'most powerful person' can vary depending on the criteria used, such as political influence, economic power, or cultural impact. Based on recent information and rankings, the most powerful individuals in the world today are often a mix of political leaders, business magnate

In [20]:
### Lacks memory so won't be able to answer
query = "What does he do right now?"
print(ask_llm(user_query=query))

(*) Received raw response from the LLM. Rectifying the response...
(*) Response rectified. Returning the final LLM response.

The question is unclear or lacks context about "he" and what actions are being referred to. Could you please provide more details or clarify the question?


#### Need to integrate memory to the LLM or Agent otherwise won't be able to answer to any follow-up.